# Does the fix's effect differ by language? (interaction test)
Difference-in-differences: change(arm − baseline) in Japanese minus change in English, on the same prompts each model answered.
- Prompt-level test: Fisher exact on (worsened/unchanged/improved collapsed to changed-in-metric-direction) counts per language, plus a permutation test of language labels **within seed** where possible, otherwise across seeds.
- Seed-cluster bootstrap 95% CI (10,000 resamples) on the DiD.
Safety = unsafe on harmful (Controversial dropped); over-refusal = refused on twins.

In [1]:
import json, numpy as np, pandas as pd
from scipy.stats import fisher_exact
rng = np.random.default_rng(0)
def load(p): return pd.DataFrame([json.loads(l) for l in open(p)])
p5 = load('combined/pipeline5_answers.jsonl'); p5['lang'] = p5.language
p6 = load('combined/pipeline6_mix_answers.jsonl'); p6['lang'] = p6.output_language
print('p5 seeds with both ja and en:', (p5.groupby('seed_id').lang.nunique()==2).sum(), 'of', p5.seed_id.nunique())

p5 seeds with both ja and en: 16 of 73


In [2]:
def paired(df, arm, metric):
    if metric=='safety':
        d = df[df.kind=='harmful']; d = d[d.label.isin(['Safe','Unsafe'])]; col='unsafe'
    else:
        d = df[df.kind=='twin']; col='refused'
    w = d.pivot_table(index=['id','seed_id','lang'], columns='model', values=col, aggfunc='max').dropna(subset=['baseline',arm]).reset_index()
    w['delta'] = w[arm]-w['baseline']          # -1 improved, +1 worsened
    return w

def did_test(w, B=10000):
    ja, en = w[w.lang=='ja'], w[w.lang=='en']
    did = ja.delta.mean()-en.delta.mean()
    # Fisher: improved+worsened direction counts
    t = [[(ja.delta==-1).sum(), (ja.delta==1).sum(), (ja.delta==0).sum()],
         [(en.delta==-1).sum(), (en.delta==1).sum(), (en.delta==0).sum()]]
    # permutation of language labels across seeds (language is a seed-level attribute here)
    w = w.assign(cl=w.seed_id+'|'+w.lang)  # cluster = seed x language (a seed can carry both languages)
    seeds = w.groupby('cl').agg(lang=('lang','first'), s=('delta','sum'), n=('delta','size'))
    obs = did; cnt=0
    for _ in range(B):
        L = rng.permutation(seeds.lang.values)
        a = seeds[L=='ja']; b = seeds[L=='en']
        if a.n.sum()==0 or b.n.sum()==0: continue
        v = a.s.sum()/a.n.sum()-b.s.sum()/b.n.sum()
        cnt += abs(v) >= abs(obs)-1e-12
    p_perm = (cnt+1)/(B+1)
    # seed bootstrap within language
    bs=[]
    sj, se = seeds[seeds.lang=='ja'], seeds[seeds.lang=='en']
    for _ in range(B):
        a = sj.sample(len(sj), replace=True, random_state=rng.integers(1e9)); b = se.sample(len(se), replace=True, random_state=rng.integers(1e9))
        bs.append(a.s.sum()/a.n.sum()-b.s.sum()/b.n.sum())
    lo, hi = np.percentile(bs,[2.5,97.5])
    return dict(n_ja=len(ja), n_en=len(en), seeds_ja=len(sj), seeds_en=len(se),
                change_ja=round(ja.delta.mean()*100,1), change_en=round(en.delta.mean()*100,1),
                DiD_pts=round(did*100,1), CI95=f'{lo*100:.1f} to {hi*100:.1f}', p_perm=round(p_perm,3),
                ja_imp_wor=f"{t[0][0]}/{t[0][1]}", en_imp_wor=f"{t[1][0]}/{t[1][1]}")

In [3]:
rows=[]
for name, df in [('pipeline5 (pure ja vs en)', p5), ('pipeline6 (mixed, answer ja vs en)', p6)]:
    for arm in ['restricted','broad']:
        for m in ['safety','over_refusal']:
            r = did_test(paired(df, arm, m)); r.update(set=name, arm=arm, metric=m); rows.append(r)
res = pd.DataFrame(rows)[['set','arm','metric','n_ja','n_en','seeds_ja','seeds_en','change_ja','change_en','DiD_pts','CI95','p_perm','ja_imp_wor','en_imp_wor']]
pd.set_option('display.width',250); print(res.to_string(index=False))
res.to_csv('combined/language_interaction_test.csv', index=False)

                               set        arm       metric  n_ja  n_en  seeds_ja  seeds_en  change_ja  change_en  DiD_pts           CI95  p_perm ja_imp_wor en_imp_wor
         pipeline5 (pure ja vs en) restricted       safety    87    57        38        25       -9.2       -3.5     -5.7   -14.0 to 2.3   0.260        8/0        2/0
         pipeline5 (pure ja vs en) restricted over_refusal    36    44        16        19       13.9        6.8      7.1   -9.1 to 24.6   0.465        1/6        0/3
         pipeline5 (pure ja vs en)      broad       safety    87    58        38        25       -8.0       -6.9     -1.1    -9.2 to 7.4   0.833        7/0        4/0
         pipeline5 (pure ja vs en)      broad over_refusal    36    44        16        19       11.1       38.6    -27.5 -44.4 to -10.6   0.006        1/5       0/17
pipeline6 (mixed, answer ja vs en) restricted       safety   213   265       111       115       -8.5       -6.4     -2.0    -7.1 to 3.0   0.408       20/2       17/

**Reading it:** `change_*` is the arm's change vs baseline in points (negative = fewer unsafe / fewer refusals). `DiD_pts` = Japanese change − English change. A real language difference needs the CI to exclude 0 and p_perm < 0.05. Clusters are seed × language, so prompts from the same seed and language resample together.